# 01 — Build the portable M12F data bundle (research machine)

Run this **locally**, in the repository that holds the research outputs (`warehouse/gold/advanced_movement/…`). Nothing is recomputed: the builder

1. **discovers** the FULL 12F sequence store and the files of the same research run (12.0 fold manifest + foundation manifest, cohort registry, 11.0 officials, 12C target, Milestone 06 TCN-B),
2. **checks** them (shapes, dtypes, channels, `meta` sha256, mask padding, fold signature vs the 12.0 foundation manifest, run mode, coverage),
3. **copies** the store byte for byte into `bundles/m12f_v1/`, writes `bundle.json` (size + sha256 of every file) and
4. **validates** the result with `open_bundle`, exactly as Colab will read it.

Same thing from a terminal: `python -m nflbowl.data.builder --repo-root . --output bundles/m12f_v1` (see `docs/nflbowl/BUNDLE_BUILD_GUIDE.md`).

Set `REPO_ROOT` if the notebook is not started from `notebooks/nflbowl/`.

In [ ]:
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").is_file() and REPO_ROOT != REPO_ROOT.parent:   # notebooks/nflbowl → repository root
    REPO_ROOT = REPO_ROOT.parent
OUT = REPO_ROOT / "bundles" / "m12f_v1"            # git-ignored
SEQUENCE_STORE = None                               # set to ".../sequence_shards/<key>" only if discovery reports several FULL stores
FOLD_MANIFEST = None                                # set only to override the 12.0 fold manifest found by discovery
REPO_ROOT, OUT

In [ ]:
# 1 — discovery: metadata only (no array is opened)
from nflbowl.data.discovery import discover_bundle_sources

sources = discover_bundle_sources(REPO_ROOT, sequence_store=SEQUENCE_STORE, fold_manifest=FOLD_MANIFEST)
print(sources.summary())

In [ ]:
# 2 — dry run: every check + the copy plan; writes nothing
from nflbowl.data.builder import build_bundle_from_repo

plan = build_bundle_from_repo(REPO_ROOT, OUT, sequence_store=SEQUENCE_STORE, fold_manifest=FOLD_MANIFEST, dry_run=True)
print(plan.text())

In [ ]:
# 3 — build (an identical existing bundle is reported up_to_date; a different one is never overwritten silently)
result = build_bundle_from_repo(REPO_ROOT, OUT, sequence_store=SEQUENCE_STORE, fold_manifest=FOLD_MANIFEST,
                                include_transfer=True, include_tcn=True, verify=True, make_zip=True)
print(result.text())

In [ ]:
# 4 — open the bundle exactly as a GPU session will (sha256 of every file)
from nflbowl.data.loading import open_bundle

bundle = open_bundle(OUT, verify=True, outer_folds=5)
bundle.summary()

## Upload

- **ZIP (recommended, one file):** upload `bundles/m12f_v1.zip` to `MyDrive/NFLBowl/`. `00_colab_quickstart.ipynb` extracts it to `MyDrive/NFLBowl/bundles/m12f_v1/` on first use.
- **Folder:** upload the whole `bundles/m12f_v1/` folder to `MyDrive/NFLBowl/bundles/m12f_v1/` (every file, including `bundle.json`).

Reports of this build: `bundles/m12f_v1.reports/` (BUILD_REPORT.md, checks.csv, plan.csv, discovery.csv / .json, build_report.json); they are not part of the bundle.